# 03b - Analise Exploratoria dos Dados (grao carreta x ano)

A EDA tem papel **central**: demonstra que a selecao das variaveis do modelo e
consequencia de uma analise exploratoria rigorosa, e nao de uma decisao arbitraria.

**Variavel resposta (Y):** `custo_ano_real` — custo anual de manutencao por carreta
(CAD/ano) em valores reais de dez/2025 (CPI Canada).

Etapas: (1) analise univariada de todas as variaveis; (2) histogramas e boxplots;
(3) relacao individual de cada variavel com Y (Pearson/Spearman, ANOVA/eta);
(4) ranking de associacao; (5) multicolinearidade (matriz de correlacao + VIF).

> **Nota metodologica.** `n_os_ano` e `custo_medio_por_os_ano` sao **componentes
> aritmeticos** do proprio Y (Y = n_os x custo medio por OS). Sao exibidos na EDA por
> completude, mas identificados como componentes: nao entram como "explicadores"
> independentes no modelo explicativo (analogo ao cuidado com denominadores).


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
TABLES = PROJECT_ROOT / "reports" / "tables"
FIGURES = PROJECT_ROOT / "reports" / "figures"
FIG_EDA = FIGURES / "eda"
FIG_EDA.mkdir(parents=True, exist_ok=True)

TARGET = "custo_ano_real"
df = pd.read_csv(DATA_PROCESSED / "base_anual_carreta_deflacionada.csv")

# componentes aritmeticos de Y (exibidos, mas sinalizados)
COMPONENTES_Y = ["n_os_ano", "custo_medio_por_os_ano"]
QUANT = [c for c in ["ano_modelo", "eixos", "comprimento", "idade_carreta",
                     "km_acumulado_fim_ano", "km_rodado_ano",
                     "n_sistemas_vmrs_distintos_ano", "share_pm_ano",
                     "n_os_ano", "custo_medio_por_os_ano",
                     "n_os_ano_anterior", "n_os_acum_ate_ano_anterior",
                     "anos_ativo_ate_ano_anterior",
                     "custo_ano_anterior", "custo_acum_ate_ano_anterior"] if c in df.columns]
QUALI = [c for c in ["cod_montadora", "flag_refrigerado", "unit_subtype", "tire_size",
                     "suspension_type", "new_used_indicator", "descricao_carreta",
                     "regiao_operacao", "provincia_estado", "vmrs_predominante_ano"]
         if c in df.columns and df[c].nunique(dropna=True) > 1]
print("N linhas:", len(df), "| carretas:", df['id_carreta'].nunique())
print("quantitativas:", len(QUANT), "| qualitativas:", len(QUALI))
print("tailgate_flag descartada por variancia nula:", df['tailgate_flag'].nunique(dropna=True) <= 1 if 'tailgate_flag' in df.columns else 'n/a')


N linhas: 49248 | carretas: 9859
quantitativas: 15 | qualitativas: 10
tailgate_flag descartada por variancia nula: True


## 1. Analise univariada — estatisticas descritivas

Para Y e cada quantitativa: N, media, mediana, desvio padrao, coeficiente de
variacao (CV), minimo, Q1, Q3, maximo, assimetria, curtose e % de ausentes.

In [2]:
def desc_quant(s):
    s = pd.to_numeric(s, errors="coerce")
    v = s.dropna()
    n = len(v)
    mean = v.mean()
    return {
        "N": n, "media": mean, "mediana": v.median(), "desvio_padrao": v.std(),
        "cv": (v.std() / mean) if mean not in (0, np.nan) and mean != 0 else np.nan,
        "min": v.min(), "Q1": v.quantile(.25), "Q3": v.quantile(.75), "max": v.max(),
        "assimetria": v.skew(), "curtose": stats.kurtosis(v) if n > 3 else np.nan,
        "pct_ausente": round(s.isna().mean()*100, 2),
    }

linhas = []
for c in [TARGET] + QUANT:
    d = desc_quant(df[c]); d["variavel"] = c
    d["papel"] = ("Y" if c == TARGET else
                  "componente de Y" if c in COMPONENTES_Y else "explicativa")
    linhas.append(d)
est = pd.DataFrame(linhas).set_index("variavel")
est = est[["papel", "N", "media", "mediana", "desvio_padrao", "cv", "min", "Q1", "Q3",
           "max", "assimetria", "curtose", "pct_ausente"]].round(3)
est.to_csv(TABLES / "03b_estatisticas_descritivas.csv")
print(est.to_string())


                                         papel      N       media     mediana  desvio_padrao     cv     min         Q1          Q3          max  assimetria  curtose  pct_ausente
variavel                                                                                                                                                                         
custo_ano_real                               Y  49248    1673.724     812.550       2400.748  1.434     0.0    317.209    2008.971    62230.924       3.786   27.597         0.00
ano_modelo                         explicativa  49242    2014.545    2015.000          5.887  0.003  1982.0   2011.000    2019.000     2026.000      -0.434   -0.517         0.01
eixos                              explicativa  49157       2.078       2.000          0.276  0.133     1.0      2.000       2.000        4.000       3.226    9.943         0.18
comprimento                        explicativa  48588      52.256      53.000          3.686  0.071    28.0   

## 2. Analise univariada — variaveis categoricas

Frequencias absolutas e relativas, numero de categorias, concentracao (share da
categoria dominante) e estatisticas de Y por categoria.

In [3]:
freq_rows, ycat_rows = [], []
for c in QUALI:
    vc = df[c].value_counts(dropna=False)
    rel = vc / vc.sum()
    freq_rows.append({"variavel": c, "n_categorias": df[c].nunique(dropna=True),
                      "categoria_dominante": str(vc.index[0]),
                      "share_dominante_pct": round(rel.iloc[0]*100, 1),
                      "pct_ausente": round(df[c].isna().mean()*100, 2)})
    for cat, g in df.groupby(c)[TARGET]:
        ycat_rows.append({"variavel": c, "categoria": str(cat), "N": int(g.size),
                          "y_media": round(g.mean(), 1), "y_mediana": round(g.median(), 1)})
freq = pd.DataFrame(freq_rows).sort_values("n_categorias", ascending=False)
freq.to_csv(TABLES / "03b_frequencia_categorias.csv", index=False)
pd.DataFrame(ycat_rows).to_csv(TABLES / "03b_y_por_categoria.csv", index=False)
print(freq.to_string(index=False))


             variavel  n_categorias categoria_dominante  share_dominante_pct  pct_ausente
    descricao_carreta           253   53' T/A PLATE VAN                 27.8         0.00
        cod_montadora            26               UTLTY                 38.5         0.02
         unit_subtype            26                  DS                 26.6         0.00
     provincia_estado            24             Ontario                 77.2         0.00
            tire_size            24             11R22.5                 66.7        15.93
vmrs_predominante_ano            23                  01                 53.5         0.00
      regiao_operacao            10                 MIS                 72.7         0.00
      suspension_type             9                 AIR                 54.6        12.98
     flag_refrigerado             2                   N                 70.5         0.00
   new_used_indicator             2                 NEW                 79.0         0.36


## 3. Histogramas e boxplots

Histogramas evidenciam concentracao, assimetria e caudas; boxplots evidenciam
dispersao, mediana, quartis e outliers. Recortes por percentil (p99) sao usados
**apenas** para legibilidade visual, sem excluir dados da analise estatistica.

In [4]:
def hist_box(s, nome, path):
    v = pd.to_numeric(s, errors="coerce").dropna()
    if v.empty:
        return
    corte = v.quantile(0.99) if v.nunique() > 20 else v.max()
    vv = v[v <= corte] if corte > v.min() else v
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    ax[0].hist(vv, bins=40, color="#3b6ea5", alpha=0.85)
    ax[0].set_title(f"Histograma — {nome}"); ax[0].set_ylabel("freq.")
    ax[1].boxplot(vv, vert=True, showfliers=True)
    ax[1].set_title(f"Boxplot — {nome}")
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

hist_box(df[TARGET], "custo_ano_real (Y)", FIG_EDA / f"quant_{TARGET}.png")
for c in QUANT:
    hist_box(df[c], c, FIG_EDA / f"quant_{c}.png")

def box_y_por_cat(c, path, topn=10):
    top = df[c].value_counts().head(topn).index
    sub = df[df[c].isin(top)]
    corte = sub[TARGET].quantile(0.99)
    grupos = [sub.loc[sub[c] == cat, TARGET].clip(upper=corte).values for cat in top]
    fig, ax = plt.subplots(figsize=(8.5, 4))
    ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)
    ax.set_title(f"Y (custo anual real) por {c} (top {topn})")
    ax.set_ylabel("CAD/ano (real)"); plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

for c in QUALI:
    box_y_por_cat(c, FIG_EDA / f"quali_{c}.png")
print("figuras geradas em reports/figures/eda/ :", len(list(FIG_EDA.glob('*.png'))))


C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)
C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)


C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)
C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)


C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)
C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)


C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)
C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)


C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)


figuras geradas em reports/figures/eda/ : 26


C:\Users\rodri\AppData\Local\Temp\ipykernel_6224\2628411275.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(grupos, labels=[str(t)[:14] for t in top], showfliers=False)


## 4. Relacao individual de cada variavel com Y

**Quantitativas:** correlacao de Pearson (linear) e Spearman (monotonica, robusta a
outliers) com Y. **Categoricas:** ANOVA (F, p-valor) e forca de associacao eta =
sqrt(SSB/SST). Cada resultado deve ser interpretado quanto a magnitude, direcao e
coerencia com o dominio (feito no texto do PowerPoint e do sumario).

In [5]:
corr_rows = []
for c in QUANT:
    x = pd.to_numeric(df[c], errors="coerce")
    m = x.notna() & df[TARGET].notna()
    if m.sum() < 100:
        continue
    pr, pp = stats.pearsonr(x[m], df[TARGET][m])
    sr, sp = stats.spearmanr(x[m], df[TARGET][m])
    corr_rows.append({"variavel": c,
                      "papel": "componente de Y" if c in COMPONENTES_Y else "explicativa",
                      "pearson": round(pr, 3), "pearson_p": round(pp, 4),
                      "spearman": round(sr, 3), "spearman_p": round(sp, 4),
                      "abs_spearman": round(abs(sr), 3), "N": int(m.sum())})
corr = pd.DataFrame(corr_rows).sort_values("abs_spearman", ascending=False)
corr.to_csv(TABLES / "03b_correlacao_com_y.csv", index=False)
print(corr.to_string(index=False))


                     variavel           papel  pearson  pearson_p  spearman  spearman_p  abs_spearman     N
       custo_medio_por_os_ano componente de Y    0.514        0.0     0.786      0.0000         0.786 49248
                     n_os_ano componente de Y    0.750        0.0     0.769      0.0000         0.769 49248
n_sistemas_vmrs_distintos_ano     explicativa    0.630        0.0     0.704      0.0000         0.704 49248
            n_os_ano_anterior     explicativa    0.590        0.0     0.540      0.0000         0.540 39389
           custo_ano_anterior     explicativa    0.588        0.0     0.536      0.0000         0.536 39389
                km_rodado_ano     explicativa    0.419        0.0     0.530      0.0000         0.530 39076
   n_os_acum_ate_ano_anterior     explicativa    0.537        0.0     0.461      0.0000         0.461 49248
  custo_acum_ate_ano_anterior     explicativa    0.548        0.0     0.452      0.0000         0.452 49248
         km_acumulado_fim_an

In [6]:
def eta_anova(c):
    sub = df[[c, TARGET]].dropna()
    grupos = [g[TARGET].values for _, g in sub.groupby(c) if len(g) >= 5]
    if len(grupos) < 2:
        return None
    F, p = stats.f_oneway(*grupos)
    grand = sub[TARGET].mean()
    ssb = sum(len(g) * (g.mean() - grand) ** 2 for g in grupos)
    sst = ((sub[TARGET] - grand) ** 2).sum()
    eta2 = ssb / sst if sst > 0 else np.nan
    return {"variavel": c, "n_categorias": sub[c].nunique(),
            "F_anova": round(F, 2), "p_valor": round(p, 6),
            "eta2": round(eta2, 4), "eta": round(np.sqrt(eta2), 4)}

eta_rows = [r for r in (eta_anova(c) for c in QUALI) if r]
eta = pd.DataFrame(eta_rows).sort_values("eta", ascending=False)
eta.to_csv(TABLES / "03b_eta_categoricas.csv", index=False)
print(eta.to_string(index=False))


             variavel  n_categorias  F_anova  p_valor   eta2    eta
    descricao_carreta           253   144.34      0.0 0.3536 0.5947
         unit_subtype            26   855.93      0.0 0.3030 0.5505
     flag_refrigerado             2 11304.55      0.0 0.1867 0.4321
vmrs_predominante_ano            23   362.98      0.0 0.1172 0.3423
        cod_montadora            26   141.09      0.0 0.0568 0.2383
            tire_size            24    48.25      0.0 0.0239 0.1546
      suspension_type             9   135.81      0.0 0.0217 0.1473
     provincia_estado            24   101.97      0.0 0.0203 0.1424
   new_used_indicator             2   716.14      0.0 0.0144 0.1199
      regiao_operacao            10    37.23      0.0 0.0060 0.0775


## 5. Ranking de associacao com Y

Ranking unico combinando quantitativas (por |Spearman|) e categoricas (por eta).
Esse ranking fundamenta a etapa posterior de **selecao das variaveis** (notebook 05).
Componentes de Y sao marcados e nao competem como explicadores.

In [7]:
r_quant = corr.assign(tipo="quantitativa (|Spearman|)",
                       forca=corr["abs_spearman"])[["variavel", "tipo", "forca", "papel"]]
r_quali = eta.assign(tipo="categorica (eta)", forca=eta["eta"], papel="explicativa")[
    ["variavel", "tipo", "forca", "papel"]]
rank = pd.concat([r_quant, r_quali], ignore_index=True).sort_values("forca", ascending=False)
rank.to_csv(TABLES / "03b_ranking_associacao.csv", index=False)

plot = rank[rank["papel"] != "componente de Y"].head(18).iloc[::-1]
cores = ["#c0563b" if "categ" in t else "#3b6ea5" for t in plot["tipo"]]
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.barh(plot["variavel"], plot["forca"], color=cores)
ax.set_xlabel("Forca de associacao com Y (|Spearman| ou eta)")
ax.set_title("Ranking de associacao com o custo anual por carreta")
fig.tight_layout(); fig.savefig(FIG_EDA / "ranking_associacao_y.png", dpi=140); plt.close(fig)
print(rank.to_string(index=False))


                     variavel                      tipo  forca           papel
       custo_medio_por_os_ano quantitativa (|Spearman|) 0.7860 componente de Y
                     n_os_ano quantitativa (|Spearman|) 0.7690 componente de Y
n_sistemas_vmrs_distintos_ano quantitativa (|Spearman|) 0.7040     explicativa
            descricao_carreta          categorica (eta) 0.5947     explicativa
                 unit_subtype          categorica (eta) 0.5505     explicativa
            n_os_ano_anterior quantitativa (|Spearman|) 0.5400     explicativa
           custo_ano_anterior quantitativa (|Spearman|) 0.5360     explicativa
                km_rodado_ano quantitativa (|Spearman|) 0.5300     explicativa
   n_os_acum_ate_ano_anterior quantitativa (|Spearman|) 0.4610     explicativa
  custo_acum_ate_ano_anterior quantitativa (|Spearman|) 0.4520     explicativa
             flag_refrigerado          categorica (eta) 0.4321     explicativa
         km_acumulado_fim_ano quantitativa (|Spearma

## 6. Multicolinearidade — matriz de correlacao e VIF

Matriz de correlacao de Spearman entre as quantitativas e VIF (Variance Inflation
Factor). Referencia usual: VIF > 5 pede atencao; VIF > 10 indica colinearidade
relevante. Em modelos de arvore/ensemble a colinearidade e menos critica que em
modelos lineares.

In [8]:
num = df[QUANT].apply(pd.to_numeric, errors="coerce")
sp = num.corr(method="spearman")
sp.to_csv(TABLES / "03b_spearman_numericas.csv")
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(sp.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(sp))); ax.set_xticklabels(sp.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(sp))); ax.set_yticklabels(sp.index, fontsize=7)
fig.colorbar(im, fraction=0.046, pad=0.04); ax.set_title("Correlacao de Spearman entre quantitativas")
fig.tight_layout(); fig.savefig(FIG_EDA / "matriz_spearman.png", dpi=140); plt.close(fig)

X = SimpleImputer(strategy="median").fit_transform(num)
X = pd.DataFrame(X, columns=QUANT)
vif_rows = []
for i, c in enumerate(QUANT):
    y = X[c]; others = X.drop(columns=[c])
    r2 = LinearRegression().fit(others, y).score(others, y)
    vif = np.inf if r2 >= 0.999 else 1.0 / (1.0 - r2)
    vif_rows.append({"variavel": c, "vif": round(vif, 2)})
vif = pd.DataFrame(vif_rows).sort_values("vif", ascending=False)
vif.to_csv(TABLES / "03b_vif.csv", index=False)
print(vif.to_string(index=False))


                     variavel   vif
                idade_carreta 13.51
   n_os_acum_ate_ano_anterior 12.87
                   ano_modelo 12.21
  custo_acum_ate_ano_anterior  9.36
            n_os_ano_anterior  5.99
           custo_ano_anterior  4.84
                     n_os_ano  3.95
  anos_ativo_ate_ano_anterior  3.05
n_sistemas_vmrs_distintos_ano  2.75
                km_rodado_ano  1.57
         km_acumulado_fim_ano  1.49
                 share_pm_ano  1.22
       custo_medio_por_os_ano  1.12
                  comprimento  1.10
                        eixos  1.05


## 7. Evolucao anual do Y (valores reais)

In [9]:
ev = df.groupby("ano")[TARGET].agg(media="mean", mediana="median",
                                    p90=lambda s: s.quantile(.9), n="size").reset_index().round(1)
ev.to_csv(TABLES / "03b_evolucao_y_anual.csv", index=False)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(ev["ano"], ev["media"], marker="o", label="Media")
ax.plot(ev["ano"], ev["mediana"], marker="s", label="Mediana")
ax.set_xlabel("Ano"); ax.set_ylabel("Custo anual por carreta (CAD real)")
ax.set_title("Evolucao do custo anual de manutencao por carreta (real, dez/2025)")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(FIG_EDA / "evolucao_y_anual.png", dpi=140); plt.close(fig)
print(ev.to_string(index=False))
print(f"\nvariacao real media 2020->2025: {(ev['media'].iloc[-1]/ev['media'].iloc[0]-1)*100:.1f}%")


 ano  media  mediana    p90    n
2020 1333.9    652.7 3179.8 6779
2021 1288.2    575.7 3209.5 7629
2022 1569.8    663.1 4145.6 8147
2023 1799.2    818.4 4668.3 8783
2024 1878.3   1023.9 4444.2 8918
2025 2025.8   1056.4 4933.6 8992

variacao real media 2020->2025: 51.9%
